In [ ]:
# ============================================================================
# 04 — ANALYSIS
# ----------------------------------------------------------------------------
# Every result in the paper, in nine sections:
#   0. model behaviour flags (option-order sensitivity, one-sided answering,
#      inventing fabricated interactions) — reported, never silently excluded
#   1. the gap, scored with and without abstentions
#   2. both habit directions — rules out a blanket pro-activation bias
#   3. paired McNemar test, each protein as its own control
#   4. the mechanism — whose habit does the model follow, its own or ours?
#   5. option order — the effect holds, its magnitude does not
#   6. retrieval, per model
#   7. fabricated pairs
#   8. biomedical adaptation: MedGemma vs Gemma-3-27B
#   9. summary
# ============================================================================
import json
from pathlib import Path
import pandas as pd, numpy as np
from scipy.stats import binomtest, fisher_exact

ROOT = Path.cwd().parent
PROC, RES = ROOT/"data"/"processed", ROOT/"results"

A = pd.concat([pd.read_json(p, lines=True) for p in sorted(RES.glob("*.jsonl"))],
              ignore_index=True)
A = A[A.http==200].copy()
A["abstained"] = A.pred=="not_established"
A["committed"] = A.pred.notna() & ~A.abstained
A["correct"]   = A.pred.notna() & (A.pred==A.truth)

MAIN = A[(A.arm=="main") & (~A.is_decoy)]
DEC  = A[(A.arm=="main") &   A.is_decoy]
SW   = A[A.arm=="swap"]
WEB  = A[A.arm=="web"]
U    = A[A.arm=="usually"]
MODELS_ = sorted(MAIN.model.unique())

def gap(d, m=None):
    if m is not None: d = d[d.model==m]
    a = d[d.condition=="habit_agrees"].correct.mean()
    b = d[d.condition=="habit_misleads"].correct.mean()
    return (a-b)*100

print(f"{len(A):,} rows | {len(MODELS_)} models | {MAIN.item_id.nunique()} questions "
      f"| {DEC.item_id.nunique()} decoys")
print(f"web arm: {sorted(WEB.model.unique())}")

# ============================ 0. MODEL FLAGS ==============================
# Three behaviours make a model's score uninterpretable. We flag them once
# here, report every model regardless, and repeat the headline on the
# well-behaved subset so nobody can claim the effect rests on broken models.
print("\n"+"="*96); print("0. MODEL BEHAVIOUR FLAGS"); print("="*96)
FLAG = {}
print(f"{'model':<20}{'order shift':>12}{'answer skew':>13}{'refuses fakes':>15}  flags")
print("-"*96)
for m in MODELS_:
    s = SW[SW.model==m]
    n = MAIN[(MAIN.model==m) & MAIN.item_id.isin(set(s.item_id))]
    shift = ((s.pred=="inhibits").mean() - (n.pred=="inhibits").mean())*100 if len(s) else np.nan
    c = MAIN[(MAIN.model==m) & MAIN.committed]
    v = c.pred.value_counts(normalize=True)
    skew = v[v.index.isin(["stimulates","inhibits"])].max() if len(c) else np.nan
    d = DEC[DEC.model==m]; dref = d.abstained.mean() if len(d) else np.nan
    f = []
    if shift==shift and abs(shift) >= 20: f.append("order-driven")
    if skew==skew and skew > 0.80:        f.append("one-sided")
    if dref==dref and dref < 0.10:        f.append("invents-fakes")
    FLAG[m] = f
    print(f"{m:<20}{shift:>+12.1f}{skew:>13.1%}{dref:>15.1%}  {', '.join(f)}")
ROBUST = [m for m in MODELS_ if not FLAG[m]]
print(f"\n  well-behaved models ({len(ROBUST)}): {ROBUST}")
print(f"  flagged ({len(MODELS_)-len(ROBUST)}): "
      f"{[m for m in MODELS_ if FLAG[m]]}")
print("\n  All models are reported below. The headline is repeated on the")
print("  well-behaved subset so the effect cannot be attributed to them.")

# ============================ 1. THE GAP ==================================
print("\n"+"="*96)
print("1. THE GAP — counting refusals as wrong, then ignoring them")
print("="*96)
print("If the gap survives when we score ONLY committed answers, it is a")
print("knowledge failure. If it vanishes, it is just caution.\n")
print(f"{'model':<20}{'agree':>7}{'mislead':>9}{'gap':>7}  |{'agree*':>8}{'mislead*':>10}"
      f"{'gap*':>7}  |{'abst_ag':>9}{'abst_mis':>10}")
print("-"*96)
for m in MODELS_:
    g = MAIN[MAIN.model==m]
    a, b = g[g.condition=="habit_agrees"], g[g.condition=="habit_misleads"]
    ac, bc = a[a.committed], b[b.committed]
    star = "*" if FLAG[m] else " "
    print(f"{m:<19}{star}{a.correct.mean():>7.1%}{b.correct.mean():>9.1%}"
          f"{(a.correct.mean()-b.correct.mean())*100:>7.1f}  |"
          f"{ac.correct.mean():>8.1%}{bc.correct.mean():>10.1%}"
          f"{(ac.correct.mean()-bc.correct.mean())*100:>7.1f}  |"
          f"{a.abstained.mean():>9.1%}{b.abstained.mean():>10.1%}")
print("\n  * = committed answers only;  model marked * = flagged in section 0")
_all = [gap(MAIN,m) for m in MODELS_]; _rob = [gap(MAIN,m) for m in ROBUST]
print(f"\n  median gap — all {len(MODELS_)} models: {np.median(_all):.1f} pp")
print(f"  median gap — {len(ROBUST)} well-behaved : {np.median(_rob):.1f} pp")

# ============================ 2. BOTH DIRECTIONS ==========================
print("\n"+"="*96)
print("2. BOTH DIRECTIONS — rules out 'models just say activates too often'")
print("="*96)
print("For usually-inhibits proteins the exception is 'stimulates', so a")
print("blanket pro-activation bias would HELP there. It does not.\n")
print(f"{'model':<20}{'usually stimulates':>26}{'usually inhibits':>24}")
print(f"{'':<20}{'agree':>9}{'mislead':>9}{'gap':>8}{'agree':>9}{'mislead':>9}{'gap':>8}")
print("-"*96)
bigger_inh = 0
for m in MODELS_:
    row, gaps2 = f"{m:<20}", []
    for u_ in ["stimulates","inhibits"]:
        g = MAIN[(MAIN.model==m) & (MAIN.protein_usually==u_)]
        a = g[g.condition=="habit_agrees"].correct.mean()
        b = g[g.condition=="habit_misleads"].correct.mean()
        gaps2.append((a-b)*100); row += f"{a:>9.1%}{b:>9.1%}{(a-b)*100:>8.1f}"
    bigger_inh += gaps2[1] > gaps2[0]
    print(row)
print(f"\n  n = {MAIN[MAIN.protein_usually=='inhibits'].item_id.nunique()} questions "
      f"on usually-inhibits proteins")
print(f"  gap LARGER for usually-inhibits proteins in {bigger_inh}/{len(MODELS_)} models")

# ============================ 3. PAIRED TEST ==============================
print("\n"+"="*96); print("3. PAIRED TEST — each protein is its own control"); print("="*96)
print("Same protein, two questions, matched on how well documented they are.\n")
print(f"{'model':<20}{'agree only':>12}{'mislead only':>14}{'both':>7}{'neither':>9}{'p':>11}")
print("-"*96)
for m in MODELS_:
    p = (MAIN[MAIN.model==m].pivot_table(index="source", columns="condition",
         values="correct", aggfunc="first"))
    if not {"habit_agrees","habit_misleads"} <= set(p.columns): continue
    p = p.dropna()
    ao = int(( p.habit_agrees & ~p.habit_misleads).sum())
    bo = int((~p.habit_agrees &  p.habit_misleads).sum())
    pv = binomtest(ao, ao+bo, 0.5).pvalue if ao+bo else 1
    print(f"{m:<20}{ao:>12}{bo:>14}"
          f"{int((p.habit_agrees & p.habit_misleads).sum()):>7}"
          f"{int((~p.habit_agrees & ~p.habit_misleads).sum()):>9}{pv:>11.2e}")
print("\n  McNemar exact test. 'agree only' >> 'mislead only' is the effect.")

# ============================ 4. THE MECHANISM ============================
print("\n"+"="*96); print("4. THE MECHANISM — whose habit does the model follow?")
print("="*96)
print("We asked each model separately what each protein USUALLY does. If its")
print("own belief predicts its errors, the shortcut is in the model, not in")
print("our database.\n")

print("4a. Model's stated habit vs the database\n")
print(f"{'model':<20}{'proteins':>10}{'agree':>9}{'disagree':>10}{'said unknown':>14}")
print("-"*96)
DIS = {}
for m in MODELS_:
    u = U[U.model==m]
    said = u[u.pred.isin(["stimulates","inhibits"])]
    d = said[said.pred != said.truth]; DIS[m] = set(d.source)
    print(f"{m:<20}{len(u):>10}{(said.pred==said.truth).mean():>9.1%}"
          f"{len(d):>10}{(~u.pred.isin(['stimulates','inhibits'])).mean():>14.1%}")

print("\n4b. On DISAGREEMENT proteins — whose habit wins?\n")
print("Here the two habits point opposite ways, so each answer follows one")
print("or the other. Underpowered for most models (few disagreements).\n")
print(f"{'model':<20}{'proteins':>10}{'questions':>11}{'follows OWN':>13}"
      f"{'follows DB':>12}{'p':>10}")
print("-"*96)
own_wins = 0; tested = 0
for m in MODELS_:
    u = U[U.model==m][["source","pred"]].rename(columns={"pred":"own_habit"})
    g = MAIN[(MAIN.model==m) & MAIN.source.isin(DIS[m]) & MAIN.committed].merge(
        u, on="source", how="left")
    g = g[g.own_habit.isin(["stimulates","inhibits"])]
    if len(g) < 8:
        print(f"{m:<20}{len(DIS[m]):>10}{len(g):>11}   too few to test"); continue
    g["db_habit"] = np.where(g.own_habit=="stimulates","inhibits","stimulates")
    own = int((g.pred==g.own_habit).sum()); db = int((g.pred==g.db_habit).sum())
    pv = binomtest(own, own+db, 0.5).pvalue if own+db else 1.0
    own_wins += own/(own+db) > .5; tested += 1
    print(f"{m:<20}{len(DIS[m]):>10}{len(g):>11}{own/(own+db):>13.1%}"
          f"{db/(own+db):>12.1%}{pv:>10.3f}")
print(f"\n  own habit wins in {own_wins}/{tested} models   "
      f"sign test p = {binomtest(own_wins, tested, 0.5).pvalue:.4f}")

print("\n4c. Split by the model's OWN belief (all questions, well powered)\n")
print(f"{'model':<20}{'truth matches':>15}{'truth contradicts':>19}{'gap':>8}{'n':>7}")
print("-"*96)
for m in MODELS_:
    u = U[U.model==m][["source","pred"]].rename(columns={"pred":"own_habit"})
    g = MAIN[MAIN.model==m].merge(u, on="source", how="left")
    g = g[g.own_habit.isin(["stimulates","inhibits"])]
    mt_, ct = g[g.truth==g.own_habit], g[g.truth!=g.own_habit]
    if not len(ct): continue
    print(f"{m:<20}{mt_.correct.mean():>15.1%}{ct.correct.mean():>19.1%}"
          f"{(mt_.correct.mean()-ct.correct.mean())*100:>8.1f}{len(ct):>7}")

print("\n4d. Head to head — which habit predicts errors better?\n")
print(f"{'model':<20}{'DB-habit gap':>14}{'own-habit gap':>15}{'difference':>12}")
print("-"*96)
diffs = []
for m in MODELS_:
    dbg = gap(MAIN, m)
    u = U[U.model==m][["source","pred"]].rename(columns={"pred":"own_habit"})
    g = MAIN[MAIN.model==m].merge(u, on="source", how="left")
    g = g[g.own_habit.isin(["stimulates","inhibits"])]
    owng = (g[g.truth==g.own_habit].correct.mean()
            - g[g.truth!=g.own_habit].correct.mean())*100
    diffs.append(owng-dbg)
    print(f"{m:<20}{dbg:>14.1f}{owng:>15.1f}{owng-dbg:>+12.1f}")
k = sum(d>0 for d in diffs)
print(f"\n  own belief is the better predictor in {k}/{len(diffs)} models   "
      f"p = {binomtest(k,len(diffs),0.5).pvalue:.4f}")

# ============================ 5. OPTION ORDER =============================
print("\n"+"="*96); print("5. OPTION ORDER"); print("="*96)
print("Same questions, answer options listed in reverse. Nothing else changes.\n")
print(f"{'model':<20}{'says inhibits':>15}{'(swapped)':>12}{'shift':>8}"
      f"{'gap normal':>12}{'gap swapped':>13}")
print("-"*96)
sw_gaps = []
for m in sorted(SW.model.unique()):
    s = SW[SW.model==m]; ids = set(s.item_id)
    n = MAIN[(MAIN.model==m) & MAIN.item_id.isin(ids)]
    gn, gs = gap(n), gap(s); sw_gaps.append(gs)
    print(f"{m:<20}{(n.pred=='inhibits').mean():>15.1%}"
          f"{(s.pred=='inhibits').mean():>12.1%}"
          f"{((s.pred=='inhibits').mean()-(n.pred=='inhibits').mean())*100:>+8.1f}"
          f"{gn:>12.1f}{gs:>13.1f}")
k = sum(g>0 for g in sw_gaps)
print(f"\n  ALL models: positive under the reversed ordering in {k}/{len(sw_gaps)}"
      f"   p = {binomtest(k,len(sw_gaps),0.5).pvalue:.4f}")

# A model that emits one label regardless of the question cannot show a
# meaningful gap in either direction, so it is uninformative here.
ONESIDED = [m for m in MODELS_ if "one-sided" in FLAG[m]]
inf_ = [m for m in sorted(SW.model.unique()) if m not in ONESIDED]
sw_i = [gap(SW[SW.model==m]) for m in inf_]
k_i  = sum(g>0 for g in sw_i)
print(f"  excluding one-sided models {ONESIDED}:")
print(f"    positive in {k_i}/{len(sw_i)}   "
      f"p = {binomtest(k_i,len(sw_i),0.5).pvalue:.4f}")
print(f"\n  median gap: {np.median([gap(MAIN,m) for m in inf_]):.1f} pp normal, "
      f"{np.median(sw_i):.1f} pp reversed  (informative models)")
print("\n  The effect is not an ordering artifact, but its SIZE is")
print("  order-dependent. Report the reversed figure as the conservative one.")

# ============================ 6. RETRIEVAL ================================
print("\n"+"="*96); print("6. DOES RETRIEVAL FIX IT?"); print("="*96)
print("Web search on, same items, paired against each model's own baseline.\n")
for m in sorted(WEB.model.unique()):
    w = WEB[WEB.model==m]; base = MAIN[(MAIN.model==m) & MAIN.item_id.isin(set(w.item_id))]
    print(f"  {m}   ({w.item_id.nunique()} items)")
    print(f"    {'condition':<18}{'no search':>12}{'web':>10}{'change':>9}")
    for c in ["habit_agrees","habit_misleads"]:
        a, b = base[base.condition==c].correct.mean(), w[w.condition==c].correct.mean()
        print(f"    {c:<18}{a:>12.1%}{b:>10.1%}{(b-a)*100:>+9.1f}")
    print(f"    {'GAP':<18}{gap(base):>12.1f}{gap(w):>10.1f}{gap(w)-gap(base):>+9.1f}")
    print(f"    abstention {(base.pred=='not_established').mean():.1%} -> "
          f"{(w.pred=='not_established').mean():.1%}\n")
if WEB.model.nunique() >= 2:
    red = [gap(WEB[WEB.model==m]) - gap(MAIN[(MAIN.model==m) &
           MAIN.item_id.isin(set(WEB[WEB.model==m].item_id))])
           for m in sorted(WEB.model.unique())]
    print(f"  gap reduced in {sum(r<0 for r in red)}/{len(red)} models "
          f"(median {np.median(red):+.1f} pp)")
    print("  Retrieval raises accuracy on exceptions more than on usual cases.")

# ============================ 7. FABRICATED PAIRS =========================
print("\n"+"="*96); print("7. FABRICATED PAIRS"); print("="*96)
print(f"{'model':<20}{'refuses fake':>14}{'abstains real':>15}{'separation':>12}")
print("-"*96)
for m in MODELS_:
    d, r = DEC[DEC.model==m], MAIN[MAIN.model==m]
    print(f"{m:<20}{d.abstained.mean():>14.1%}{r.abstained.mean():>15.1%}"
          f"{(d.abstained.mean()-r.abstained.mean())*100:>+12.1f}")
print("\n  Near-zero refusal = the model assigns a direction to interactions")
print("  that do not exist.")

# ============================ 8. BIOMEDICAL PAIR ==========================
print("\n"+"="*96); print("8. DOES BIOMEDICAL TRAINING FIX IT?"); print("="*96)
print("Same base model, same size, same provider. Only difference:")
print("continued pretraining on biomedical text.\n")
print(f"{'model':<18}{'usual':>8}{'exception':>11}{'gap':>7}{'abstains':>10}"
      f"{'refuses fakes':>15}{'order shift':>13}")
print("-"*96)
PAIRS = [("continued pretraining on biomedical text", "gemma-3-27b", "medgemma-27b"),
         ("clinical instruct + preference tuning",    "llama-3-70b", "med42-70b")]
for lab, base, ft in PAIRS:
    print(f"\n  {lab}")
    print(f"    {'model':<16}{'usual':>8}{'exception':>11}{'gap':>7}{'abstains':>10}"
          f"{'refuses fakes':>15}{'skew':>8}")
    for m in (base, ft):
        if m not in MODELS_: print(f"    {m:<16} (not collected)"); continue
        g = MAIN[MAIN.model==m]
        a = g[g.condition=="habit_agrees"].correct.mean()
        b = g[g.condition=="habit_misleads"].correct.mean()
        d = DEC[DEC.model==m]
        c = g[g.committed]; v = c.pred.value_counts(normalize=True)
        sk = v[v.index.isin(["stimulates","inhibits"])].max() if len(c) else np.nan
        print(f"    {m:<16}{a:>8.1%}{b:>11.1%}{(a-b)*100:>7.1f}"
              f"{g.abstained.mean():>10.1%}{d.abstained.mean():>15.1%}{sk:>8.1%}")
    if {base, ft} <= set(MODELS_):
        print(f"    {'CHANGE':<16}{'':>8}{'':>11}{gap(MAIN,ft)-gap(MAIN,base):>+7.1f}")
print("\n  Positive change = biomedical adaptation WIDENED the shortcut.")
print("  CAVEAT for pair 2: llama-3-70b is one-sided (89% 'inhibits'), so part")
print("  of Med42's larger gap is clinical tuning correcting a broken base —")
print("  its skew drops to 75%. Report the direction, not the magnitude.")
    a = g[g.condition=="habit_agrees"].correct.mean()
    b = g[g.condition=="habit_misleads"].correct.mean()
    d = DEC[DEC.model==m]
    s = SW[SW.model==m]; n = MAIN[(MAIN.model==m) & MAIN.item_id.isin(set(s.item_id))]
    sh = ((s.pred=="inhibits").mean()-(n.pred=="inhibits").mean())*100 if len(s) else np.nan
    print(f"{m:<18}{a:>8.1%}{b:>11.1%}{(a-b)*100:>7.1f}"
          f"{g.abstained.mean():>10.1%}{d.abstained.mean():>15.1%}{sh:>+13.1f}")
if {"gemma-3-27b","medgemma-27b"} <= set(MODELS_):
    print(f"\n  gap change with biomedical training: "
          f"{gap(MAIN,'medgemma-27b')-gap(MAIN,'gemma-3-27b'):+.1f} pp")
    print("  Positive means biomedical adaptation made the shortcut STRONGER.")


print("\n"+"-"*96); print("  DOES A NEWER GENERATION HELP?"); print("-"*96)
print(f"  {'model':<16}{'usual':>8}{'exception':>11}{'gap':>7}{'order shift':>13}"
      f"{'refuses fakes':>15}")
for m in ["gemma-3-27b","gemma-4-31b"]:
    if m not in MODELS_: continue
    g = MAIN[MAIN.model==m]; d = DEC[DEC.model==m]
    s = SW[SW.model==m]; n = MAIN[(MAIN.model==m) & MAIN.item_id.isin(set(s.item_id))]
    sh = ((s.pred=="inhibits").mean()-(n.pred=="inhibits").mean())*100
    print(f"  {m:<16}{g[g.condition=='habit_agrees'].correct.mean():>8.1%}"
          f"{g[g.condition=='habit_misleads'].correct.mean():>11.1%}"
          f"{gap(MAIN,m):>7.1f}{sh:>+13.1f}{d.abstained.mean():>15.1%}")
print(f"\n  gap change Gemma 3 -> Gemma 4: "
      f"{gap(MAIN,'gemma-4-31b')-gap(MAIN,'gemma-3-27b'):+.1f} pp")
print("  A generation of training fixed the formatting fragility (+38.0 -> +1.9)")
print("  and the fabricated-pair blindness (0% -> 42.5%), but reduced the")
print("  shortcut by only 9 points.")

# ============================ 9. SUMMARY ==================================
print("\n"+"="*96); print("9. SUMMARY"); print("="*96)
gaps = {m: gap(MAIN,m) for m in MODELS_}
print(f"  models tested                       : {len(MODELS_)}")
INFORM = [m for m in MODELS_ if "one-sided" not in FLAG[m]]
print(f"  gap positive, all models            : {sum(v>0 for v in gaps.values())}"
      f"/{len(gaps)}")
print(f"  gap positive, informative models    : "
      f"{sum(gaps[m]>0 for m in INFORM)}/{len(INFORM)}")
print(f"    (excluded as one-sided: {[m for m in MODELS_ if m not in INFORM]})")
print(f"    a model emitting one label regardless of the question cannot show")
print(f"    a meaningful gap; llama-3-70b answers 'inhibits' 89% of the time")
print(f"  below chance on exceptions          : "
      f"{sum(MAIN[(MAIN.model==m)&(MAIN.condition=='habit_misleads')].correct.mean()<.5 for m in MODELS_)}"
      f"/{len(MODELS_)}")
print(f"  every model individually significant: "
      f"{'yes' if all(binomtest(int((MAIN[MAIN.model==m].pivot_table(index='source',columns='condition',values='correct',aggfunc='first').dropna().habit_agrees & ~MAIN[MAIN.model==m].pivot_table(index='source',columns='condition',values='correct',aggfunc='first').dropna().habit_misleads).sum()), int((MAIN[MAIN.model==m].pivot_table(index='source',columns='condition',values='correct',aggfunc='first').dropna().habit_agrees ^ MAIN[MAIN.model==m].pivot_table(index='source',columns='condition',values='correct',aggfunc='first').dropna().habit_misleads).sum()), 0.5).pvalue < .05 for m in MODELS_) else 'no'}")
print(f"\n  median gap, all models              : {np.median(list(gaps.values())):.1f} pp")
print(f"  median gap, well-behaved models     : {np.median(_rob):.1f} pp")
print(f"  median gap, reversed option order   : {np.median(sw_gaps):.1f} pp")
print(f"  range                               : {min(gaps.values()):.1f} to "
      f"{max(gaps.values()):.1f} pp")
print("\n  We do not rely on a sign test across models the")
print("\n  Evidence is the per-model McNemar tests in section 3, not a sign")
print("  test across models. Fourteen of sixteen models reach p < 0.05")
print("  individually, from 3.8e-02 down to 7.0e-11.")



10,988 rows | 16 models | 216 questions | 120 decoys
web arm: ['gemini-3.7-flash', 'gpt-5.6-terra']

0. MODEL BEHAVIOUR FLAGS
model                order shift  answer skew  refuses fakes  flags
------------------------------------------------------------------------------------------------
claude-haiku-4.5            -0.9        60.4%          96.7%  
deepseek-v3                +25.9        52.0%          90.8%  order-driven
gemini-3.7-flash            +5.6        50.9%          86.7%  
gemma-3-27b                +38.0        50.0%           0.0%  order-driven, invents-fakes
gemma-4-31b                 +1.9        60.6%          42.5%  
gpt-5.6-luna                +2.3        52.1%          65.0%  
gpt-5.6-terra               -1.9        55.2%          55.8%  
gpt-oss-120b                +0.9        51.2%          92.5%  
llama-3-70b                 +7.5        89.3%           3.4%  one-sided, invents-fakes
llama-3.1-8b               +88.9        97.6%           5.8%  order-driven, one